In [ ]:
import pandas as pd

df = pd.read_csv("/content/movies_metadata.csv")

df = df.dropna(subset=["title", "overview"])
texts = (df["title"] + ". " + df["overview"]).tolist()

print(f"Loaded {len(texts)} combined movie title+overview entries.")

Loaded 44506 combined movie title+overview entries.


In [ ]:
!pip install torch torchvision transformers accelerate bitsandbytes safetensors huggingface_hub
!pip install optimum
!pip install transformers sentencepiece protobuf accelerate huggingface-hub

!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

!make
%cd ..
!pip install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt


fatal: destination path 'llama.cpp' already exists and is not an empty directory.
/content/llama.cpp
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
/content
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
  Cloning https://github.com/huggingface/transformers (to revision v4.56.0-Embedding-Gemma-preview) to /tmp/pip-req-build-hjgqcfl2
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-hjgqcfl2
  Running command git checkout -q 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Resolved https://github.com/huggingface/transformers to commit 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
I

In [ ]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(repo_id="Gryphe/MythoMist-7B", local_dir="./MythoMist-7B")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.36.2  # This version definitely has Mistral

Found existing installation: transformers 4.57.0.dev0
Uninstalling transformers-4.57.0.dev0:
  Successfully uninstalled transformers-4.57.0.dev0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "./MythoMist-7B"


# 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=None,
    torch_dtype=torch.float16,
    trust_remote_code=True
).to('cpu')  # Explicitly move to CPU
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
import torch.nn as nn
activations = {}

def get_activation_hook(name):
    def hook(model, input, output):
        if isinstance(output, torch.Tensor):
            mean_act = output.detach().abs().mean(dim=(0, 1)).cpu()
            activations[name] = activations.get(name, torch.zeros_like(mean_act)) + mean_act
    return hook

hooks = []
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        hooks.append(module.register_forward_hook(get_activation_hook(name)))

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader

batch_size = 32
dataset = texts
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

for batch in tqdm(dataloader, desc="Analyzing activations"):
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        model(**inputs)


for h in hooks:
    h.remove()

Analyzing activations: 100%|██████████| 1391/1391 [2:32:54<00:00,  6.60s/it]


In [ ]:
torch.save(activations, "activations.pt")

In [ ]:
import torch
import torch.nn as nn

activations = torch.load("activations.pt", map_location='cpu')
threshold = 0.9

print("Processing activations...")
for name, act in list(activations.items()):
    act = act.cpu().float()
    cutoff = torch.quantile(act, threshold)
    mask = (act < cutoff)  # Keep as boolean mask
    activations[name] = mask
    print(f"✓ {name}: {act.shape} -> {mask.shape}")

print("\nApplying to model (Q8 compatible)...")
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) and name in activations:
        try:
            mask = activations[name].to(module.weight.device)

            # Handle mask shapes for Q8 weights
            weight_shape = module.weight.shape
            if mask.dim() == 1:
                if mask.size(0) == weight_shape[1]:  # Output features
                    mask = mask.unsqueeze(0).expand(weight_shape[0], -1)
                elif mask.size(0) == weight_shape[0]:  # Input features
                    mask = mask.unsqueeze(1).expand(-1, weight_shape[1])
                else:
                    print(f"  Shape mismatch: {name}")
                    continue

            print(f"Pruning {name}: weight {weight_shape}, zeroing {(~mask).sum().item()} elements")

            with torch.no_grad():
                # For Q8 models, we need to work with the dequantized weights
                if hasattr(module, 'weight'):
                    # Zero out pruned weights directly (this should work with Q8)
                    module.weight.data[~mask] = 0

            print(f"✓ Applied mask to {name}")

        except Exception as e:
            print(f"✗ Error applying mask to {name}: {e}")

Processing activations...
✓ model.layers.0.self_attn.q_proj: torch.Size([4096]) -> torch.Size([4096])
✓ model.layers.0.self_attn.k_proj: torch.Size([1024]) -> torch.Size([1024])
✓ model.layers.0.self_attn.v_proj: torch.Size([1024]) -> torch.Size([1024])
✓ model.layers.0.self_attn.o_proj: torch.Size([4096]) -> torch.Size([4096])
✓ model.layers.0.mlp.gate_proj: torch.Size([14336]) -> torch.Size([14336])
✓ model.layers.0.mlp.up_proj: torch.Size([14336]) -> torch.Size([14336])
✓ model.layers.0.mlp.down_proj: torch.Size([4096]) -> torch.Size([4096])
✓ model.layers.1.self_attn.q_proj: torch.Size([4096]) -> torch.Size([4096])
✓ model.layers.1.self_attn.k_proj: torch.Size([1024]) -> torch.Size([1024])
✓ model.layers.1.self_attn.v_proj: torch.Size([1024]) -> torch.Size([1024])
✓ model.layers.1.self_attn.o_proj: torch.Size([4096]) -> torch.Size([4096])
✓ model.layers.1.mlp.gate_proj: torch.Size([14336]) -> torch.Size([14336])
✓ model.layers.1.mlp.up_proj: torch.Size([14336]) -> torch.Size([14336

In [ ]:
model.save_pretrained("/content/MythoMist-7B-pruned-movies10")
tokenizer.save_pretrained("/content/MythoMist-7B-pruned-movies10")

('/content/MythoMist-7B-pruned-movies10/tokenizer_config.json',
 '/content/MythoMist-7B-pruned-movies10/special_tokens_map.json',
 '/content/MythoMist-7B-pruned-movies10/tokenizer.model',
 '/content/MythoMist-7B-pruned-movies10/added_tokens.json',
 '/content/MythoMist-7B-pruned-movies10/tokenizer.json')

In [ ]:
import shutil
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

source_path = '/content/MythoMist-7B-pruned-movies10'
destination_path = '/content/drive/MyDrive/MythoMist-7B-pruned-movies10'

try:
    if os.path.exists(source_path):
        print(f"Source folder found: {source_path}")

        if os.path.exists(destination_path):
            print(f"Destination folder already exists. Overwriting...")
            shutil.rmtree(destination_path)

        shutil.copytree(source_path, destination_path)
        print(f"Successfully copied folder to: {destination_path}")

        if os.path.exists(destination_path):
            print("Copy verification: SUCCESS")
        else:
            print("Copy verification: FAILED")

    else:
        print(f"Error: Source folder not found at {source_path}")
        print("Available files/folders in /content:")
        print(os.listdir('/content'))

except Exception as e:
    print(f"Error occurred: {str(e)}")

Mounted at /content/drive
Source folder found: /content/MythoMist-7B-pruned-movies10
Successfully copied folder to: /content/drive/MyDrive/MythoMist-7B-pruned-movies10
Copy verification: SUCCESS


In [ ]:
!mkdir -p output

# conversion
!python llama.cpp/convert_hf_to_gguf.py /content/MythoMist-7B-pruned-movies10 --outfile output/mythomist-7b.q8_0.gguf --outtype q8_0

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
INFO:hf-to-gguf:Loading model: MythoMist-7B-pruned-movies10
INFO:hf-to-gguf:Model architecture: MistralForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00

In [ ]:
!./llama.cpp/quantize output/mythomist-7b.q8.gguf output/mythomist-7b.pruned.q5_k_m.gguf q5_k_m

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/drive')

source_folder = "/content"
destination_folder = "/drive/MyDrive/your_folder"

if os.path.exists(destination_folder):
    shutil.rmtree(destination_folder)

shutil.copytree(source_folder, destination_folder)
print(f"Folder copied to: {destination_folder}")

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


KeyboardInterrupt: 

In [ ]:
import gc
del model
del tokenizer
gc.collect()

163

In [ ]:
HF_PRUNED_DIR="/content/MythoMist-7B-pruned-movies10"
OUT_F16_GGUF="/content/MythoMist-7B-pruned-movies10.gguf"


In [ ]:
!pip install gguf
import gc
import torch
import gguf
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np

def convert_to_gguf_complete(hf_path, gguf_path):

    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        hf_path,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(hf_path, trust_remote_code=True)
    config = model.config

    # Initialize GGUF writer
    print("Initializing GGUF writer...")
    gguf_writer = gguf.GGUFWriter(gguf_path, "llama")

    # the LLAMA-SPECIFIC keys with correct data types, careful w em:
    gguf_writer.add_uint32('llama.context_length', 8192)
    gguf_writer.add_uint32('llama.embedding_length', config.hidden_size)
    gguf_writer.add_uint32('llama.block_count', config.num_hidden_layers)
    gguf_writer.add_uint32('llama.feed_forward_length', getattr(config, 'intermediate_size', config.hidden_size * 4))

    # Attention parameters
    gguf_writer.add_uint32('llama.attention.head_count', config.num_attention_heads)
    gguf_writer.add_uint32('llama.attention.head_count_kv', getattr(config, 'num_key_value_heads', config.num_attention_heads))
    gguf_writer.add_float32('llama.attention.layer_norm_rms_epsilon', getattr(config, 'rms_norm_eps', 1e-6))  # ← FIXED: float32 not uint32

    # Rope parameters
    gguf_writer.add_uint32('llama.rope.dimension_count', config.hidden_size // config.num_attention_heads)
    gguf_writer.add_float32('llama.rope.freq_base', getattr(config, 'rope_theta', 10000.0))

    # Standard fields
    gguf_writer.add_uint32('vocab_size', config.vocab_size)
    gguf_writer.add_uint32('hidden_size', config.hidden_size)
    gguf_writer.add_uint32('num_attention_heads', config.num_attention_heads)
    gguf_writer.add_uint32('num_hidden_layers', config.num_hidden_layers)
    gguf_writer.add_uint32('num_key_value_heads', getattr(config, 'num_key_value_heads', config.num_attention_heads))
    gguf_writer.add_float32('rms_norm_eps', getattr(config, 'rms_norm_eps', 1e-6))
    gguf_writer.add_float32('rope_theta', getattr(config, 'rope_theta', 10000.0))


    gguf_writer.add_tokenizer_model('llama')

    vocab = tokenizer.get_vocab()
    tokens = list(vocab.keys())
    scores = [0.0] * len(tokens)
    toktypes = [1] * len(tokens)

    gguf_writer.add_token_list(tokens)
    gguf_writer.add_token_scores(scores)
    gguf_writer.add_token_types(toktypes)

    # Write model tensors
    print("Writing model tensors...")
    for name, param in model.named_parameters():
        if param.requires_grad:
            data = param.data.float().numpy()
            gguf_writer.add_tensor(name, data)

    # Finish up and close
    print("Finalizing GGUF file...")
    gguf_writer.write_header_to_file()
    gguf_writer.write_kv_data_to_file()
    gguf_writer.write_tensors_to_file()
    gguf_writer.close()

    print(f"Complete GGUF conversion: {gguf_path}")

In [ ]:
convert_to_gguf_complete(HF_PRUNED_DIR, OUT_F16_GGUF)

Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Initializing GGUF writer...
Writing model tensors...
Finalizing GGUF file...
Complete GGUF conversion: /content/MythoMist-7B-pruned-movies01.gguf


In [ ]:
%cd /content
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

!mkdir -p build
%cd build
!cmake ..
!cmake --build . --config Release -j

/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 65337, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 65337 (delta 5), reused 1 (delta 1), pack-reused 65322 (from 2)
Receiving objects: 100% (65337/65337), 181.10 MiB | 39.42 MiB/s, done.
Resolving deltas: 100% (47535/47535), done.
/content/llama.cpp
/content/llama.cpp/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1") 
-- The

In [ ]:
INPUT_GGUF = "/content/MythoMist-7B-pruned-movies40.gguf"
OUTPUT_Q5_KM = "/content/MythoMist-7B-pruned-movies40-Q5_K_M.gguf"

!/content/llama.cpp/build/bin/llama-quantize "$INPUT_GGUF" "$OUTPUT_Q5_KM" q5_k_m

# Verify if successful
import os
if os.path.exists(OUTPUT_Q5_KM):
    print("Quantization successful!")
    size = os.path.getsize(OUTPUT_Q5_KM) / (1024**3)
    print(f"Output: {OUTPUT_Q5_KM}")
    print(f"Size: {size:.2f} GB")
else:
    print("Quantization failed")

main: build = 6816 (03792ad9)
main: built with cc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0 for x86_64-linux-gnu
main: quantizing '/content/MythoMist-7B-pruned-F16.gguf' to '/content/MythoMist-7B-pruned-Q5_K_M.gguf' as Q5_K_M
llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /content/MythoMist-7B-pruned-F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                       llama.context_length u32              = 8192
llama_model_loader: - kv   2:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   3:                          llama.block_count u32              = 32
llama_model_loader: - kv   4:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   5:                 